# Baseline: TF-IDF + Logistic Regression

负责人：成员 1。输入为 clean_v2 的 `text_norm`。

2026-11-04：已从 Member 4 的 OOF 折号记录恢复统一划分，文件 SHA-256 与其原始记录完全一致；来源见 `member1/split_provenance.json`。

本 Notebook 先检查数据和环境，检查通过后运行正式五折 baseline。每次执行创建新 run_id。

- TF-IDF 仅在各训练折内拟合，评估使用开发集 OOF。
- 测试集推理与评分延后，预测缓存里的 test arrays 为空。
- 后续调参和 SVM 尚未完成。


In [ ]:
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this Notebook inside the project folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from member1.baseline import readiness, run
from src.data import load_clean
import pandas as pd
from IPython.display import display, Image

status = readiness()
display(status)
clean = load_clean()
display(clean.groupby(["label", "label_name"]).size().rename("samples").to_frame())

## Experiment configuration

`MODE="team"` 使用正式 Data Splits 和团队 Environment。

仅需调试时可手动设为 `MODE="provisional"`，临时 Results 只保存到 `member1/outputs/`，不会进入团队 Results 或 Prediction Cache。

每次运行都会生成新的 run_id。TF-IDF 在每个 Training Fold 内独立 fit；同一 group_id 不跨 Fold。

In [ ]:
MODE = "team"
output = None
if MODE == "team" and not status["ready_for_team_run"]:
    print("Team Run pending. Check input_errors and environment_mismatches above.")
else:
    output = run(mode=MODE)
    print(output)

In [ ]:
if output is not None:
    import json
    summary = json.loads((output / "summary.json").read_text())
    display(pd.DataFrame({"CV mean": summary["cv_mean"], "CV standard deviation": summary["cv_std_sample"]}))
    display(pd.read_csv(output / "classification_report.csv", index_col=0))
    display(Image(filename=str(output / "confusion_matrix.png")))
    display(Image(filename=str(output / "top_words.png")))

## Interpretation and reporting

- `results/baseline.csv` 的团队 Metrics 使用 pooled OOF Predictions；各 Fold 的 mean 和 sample standard deviation 另存于本次输出目录。
- Confusion Matrix 的 rows 为 True Labels，columns 为 Predicted Labels。
- Top Words 按各 Class 的正向 LR Coefficients 排列，不是 Word Frequency，也不能解释为 Causality。
- Fit time 为五个 Training Folds 的 TF-IDF + LR fit 总时间，不含 Plotting、Inference 或解释用 Refit。
- Methodology 初稿见 `docs/report_drafts/m1_baseline.md`；实验记录见 `docs/experiments/m1.md`。
- 最终单个 Notebook 合并时，需要将 `member1/baseline.py` 的函数一并内联，避免依赖遗漏。
